# Clustering de comportamiento de consumo eléctrico por cliente

## Objetivo

Esta libreta construye una segmentación de comportamiento usando los datos originales de consumo cada media hora (`hhblock_dataset`).

El objetivo no es clasificar cuánto consume un cliente, porque esa parte corresponde al modelo de magnitud bajo/medio/alto. Aquí se busca identificar **cómo consume** cada hogar:

- Consumo nocturno, matutino, vespertino o distribuido.
- Mayor uso en fin de semana o entre semana.
- Clientes con picos de consumo o comportamiento atípico.

El resultado esperado es una tabla por cliente con etiquetas interpretables, por ejemplo:

> Casa 1: bajo consumo, nocturno, fin de semana y atípico.

Esta libreta está pensada para integrarse después con el modelo de magnitud ya construido.


# 1. Configuración general

Se importan librerías, se definen rutas del proyecto y se crean carpetas de salida para tablas y gráficas.


In [1]:
from pathlib import Path
import os
import warnings

os.environ.setdefault("MPLCONFIGDIR", str((Path("./.matplotlib_cache")).resolve()))
os.environ.setdefault("XDG_CACHE_HOME", str((Path("./.cache")).resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
ORIGINAL_BASE = DATA_DIR / "original" / "kagglehub" / "datasets" / "jeanmidev" / "smart-meters-in-london" / "versions" / "21"
HHBLOCK_DIR = ORIGINAL_BASE / "hhblock_dataset" / "hhblock_dataset"
HOUSEHOLDS_PATH = ORIGINAL_BASE / "informations_households.csv"

OUTPUT_DIR = PROJECT_DIR / "output" / "clustering_comportamiento_final"
IMAGES_DIR = OUTPUT_DIR / "images"
TABLES_DIR = OUTPUT_DIR / "tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

print("Directorio del proyecto:", PROJECT_DIR)
print("Directorio hhblock:", HHBLOCK_DIR)
print("Existe hhblock:", HHBLOCK_DIR.exists())
print("Archivo households:", HOUSEHOLDS_PATH)
print("Existe households:", HOUSEHOLDS_PATH.exists())


Directorio del proyecto: /Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics
Directorio hhblock: /Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehub/datasets/jeanmidev/smart-meters-in-london/versions/21/hhblock_dataset/hhblock_dataset
Existe hhblock: True
Archivo households: /Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehub/datasets/jeanmidev/smart-meters-in-london/versions/21/informations_households.csv
Existe households: True


# 2. Inventario de archivos originales

Se valida que existan los bloques `block_*.csv` del dataset de media hora. Estos archivos contienen una fila por hogar y día, con 48 columnas de consumo: `hh_0` a `hh_47`. Cada columna representa un intervalo de 30 minutos.


In [2]:
hhblock_files = sorted(HHBLOCK_DIR.glob("block_*.csv"))

inventario = pd.DataFrame({
    "archivo": [p.name for p in hhblock_files],
    "ruta": [str(p) for p in hhblock_files],
    "tamano_mb": [round(p.stat().st_size / (1024 ** 2), 2) for p in hhblock_files]
})

print("Número de bloques hhblock encontrados:", len(hhblock_files))
display(inventario.head())
display(inventario.tail())


Número de bloques hhblock encontrados: 112


,archivo,ruta,tamano_mb
0,block_0.csv,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...,11.4400
1,block_1.csv,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...,14.0800
2,block_10.csv,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...,14.3200
3,block_100.csv,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...,14.6800
4,block_101.csv,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...,13.6000


,archivo,ruta,tamano_mb
107,block_95.csv,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...,14.8100
108,block_96.csv,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...,14.9300
109,block_97.csv,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...,14.7800
110,block_98.csv,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...,15.6600
111,block_99.csv,/Users/malgeak/Documents/Anahuac/Machine Learning/Proyecto Final/github/SmartEnergy_Analytics/data/original/kagglehu...,15.3700


# 3. Inspección de esquema

Antes de procesar todos los bloques, se revisa un archivo de muestra para confirmar las columnas disponibles.


In [3]:
if not hhblock_files:
    raise FileNotFoundError(f"No se encontraron archivos block_*.csv en {HHBLOCK_DIR}")

muestra = pd.read_csv(hhblock_files[0], nrows=5)
hh_cols = [col for col in muestra.columns if col.startswith("hh_")]

print("Columnas de la muestra:")
print(muestra.columns.tolist())
print("Número de columnas hh:", len(hh_cols))
display(muestra.head())


Columnas de la muestra:
['LCLid', 'day', 'hh_0', 'hh_1', 'hh_2', 'hh_3', 'hh_4', 'hh_5', 'hh_6', 'hh_7', 'hh_8', 'hh_9', 'hh_10', 'hh_11', 'hh_12', 'hh_13', 'hh_14', 'hh_15', 'hh_16', 'hh_17', 'hh_18', 'hh_19', 'hh_20', 'hh_21', 'hh_22', 'hh_23', 'hh_24', 'hh_25', 'hh_26', 'hh_27', 'hh_28', 'hh_29', 'hh_30', 'hh_31', 'hh_32', 'hh_33', 'hh_34', 'hh_35', 'hh_36', 'hh_37', 'hh_38', 'hh_39', 'hh_40', 'hh_41', 'hh_42', 'hh_43', 'hh_44', 'hh_45', 'hh_46', 'hh_47']
Número de columnas hh: 48


,LCLid,day,hh_0,hh_1,hh_2,hh_3,hh_4,hh_5,hh_6,hh_7,hh_8,hh_9,hh_10,hh_11,hh_12,hh_13,hh_14,hh_15,hh_16,hh_17,hh_18,hh_19,hh_20,hh_21,hh_22,hh_23,hh_24,hh_25,hh_26,hh_27,hh_28,hh_29,hh_30,hh_31,hh_32,hh_33,hh_34,hh_35,hh_36,hh_37,hh_38,hh_39,hh_40,hh_41,hh_42,hh_43,hh_44,hh_45,hh_46,hh_47
0,MAC000002,2012-10-13,0.2630,0.2690,0.2750,0.2560,0.2110,0.1360,0.1610,0.1190,0.1670,0.1090,0.1680,0.1070,0.1660,0.1170,0.1570,0.1260,0.1460,0.1060,0.1350,0.1910,0.9150,0.9330,0.1220,0.1380,0.0760,0.1330,0.0760,0.1330,0.0850,0.2630,0.1340,0.2350,0.1240,0.1840,0.2300,0.1760,0.3880,0.2600,0.9180,0.2780,0.2670,0.2390,0.2300,0.2330,0.2350,0.1880,0.2590,0.2500
1,MAC000002,2012-10-14,0.2620,0.1660,0.2260,0.0880,0.1260,0.0820,0.1230,0.0830,0.1200,0.0790,0.1210,0.0750,0.1240,0.0730,0.1250,0.0700,0.1300,0.1080,0.1960,0.3460,0.5240,0.0760,0.1290,0.6670,0.2300,0.2200,0.1630,0.0910,0.1700,0.1100,0.1100,0.1210,0.0990,0.1570,0.0930,0.3710,0.3860,1.0850,1.0750,0.9560,0.8210,0.7450,0.7120,0.5110,0.2310,0.2100,0.2780,0.1590
2,MAC000002,2012-10-15,0.1920,0.0970,0.1410,0.0830,0.1320,0.0700,0.1300,0.0740,0.1240,0.0780,0.1180,0.0820,0.1120,0.0870,0.1060,0.1400,0.1200,1.0750,0.1460,0.1230,0.0820,0.1270,0.0770,0.5510,0.1490,0.1290,0.0750,0.1300,0.0750,0.1290,0.0750,0.1280,0.1660,0.1940,0.6950,0.2600,0.2270,0.2550,1.1640,0.2490,0.2250,0.2580,0.2600,0.3340,0.2990,0.2360,0.2410,0.2370
3,MAC000002,2012-10-16,0.2370,0.2370,0.1930,0.1180,0.0980,0.1070,0.0940,0.1090,0.0910,0.1050,0.0910,0.1040,0.0920,0.1030,0.0930,0.1010,0.1440,0.1000,0.4080,0.1020,0.1000,0.1160,0.3540,0.1460,0.1900,0.9910,0.3100,0.1210,0.1130,0.0940,0.1190,0.0870,0.1300,0.2380,0.2040,0.2840,0.4470,0.2660,0.9660,0.1720,0.1920,0.2280,0.2030,0.2110,0.1880,0.2130,0.1570,0.2020
4,MAC000002,2012-10-17,0.1570,0.2110,0.1550,0.1690,0.1010,0.1170,0.0840,0.1180,0.0800,0.1190,0.0750,0.1230,0.0710,0.1260,0.0670,0.1240,0.1180,0.1320,0.3580,0.6280,0.7840,0.6810,0.7490,0.5930,0.5020,0.1150,0.1130,0.0920,0.1240,0.0840,0.1250,0.0780,0.1360,0.2270,0.2070,0.1410,0.2580,0.2170,0.2230,0.0750,0.2300,0.2080,0.2650,0.3770,0.3270,0.2770,0.2880,0.2560


# 4. Diseño de variables de comportamiento

El modelo de comportamiento debe evitar que el volumen total domine la segmentación. Por eso se usarán principalmente proporciones y razones.

Las variables se agrupan en cuatro familias:

1. **Distribución horaria:** porcentaje de consumo en madrugada, mañana, tarde y noche.
2. **Tipo de día:** porcentaje de consumo en fin de semana.
3. **Regularidad:** variabilidad del consumo diario.
4. **Picos:** intensidad del mayor consumo horario frente al consumo promedio.

Estas variables permiten describir clientes como nocturnos, de fin de semana, estables, irregulares o atípicos.


In [4]:
# Mapeo de las 48 medias horas a hora del día
hh_index = pd.Series(hh_cols).str.replace("hh_", "", regex=False).astype(int)
hh_to_hour = dict(zip(hh_cols, (hh_index // 2).astype(int)))

period_cols = {
    "madrugada": [col for col in hh_cols if hh_to_hour[col] < 6],
    "manana": [col for col in hh_cols if 6 <= hh_to_hour[col] < 12],
    "tarde": [col for col in hh_cols if 12 <= hh_to_hour[col] < 18],
    "noche": [col for col in hh_cols if hh_to_hour[col] >= 18],
}

for periodo, cols in period_cols.items():
    print(periodo, len(cols), cols[:3], "...", cols[-3:])


madrugada 12 ['hh_0', 'hh_1', 'hh_2'] ... ['hh_9', 'hh_10', 'hh_11']
manana 12 ['hh_12', 'hh_13', 'hh_14'] ... ['hh_21', 'hh_22', 'hh_23']
tarde 12 ['hh_24', 'hh_25', 'hh_26'] ... ['hh_33', 'hh_34', 'hh_35']
noche 12 ['hh_36', 'hh_37', 'hh_38'] ... ['hh_45', 'hh_46', 'hh_47']


# 5. Función de agregación por bloque

Para evitar consumir demasiada memoria, no se convierte todo el dataset a formato largo. En su lugar, cada bloque se procesa de forma agregada y después se consolida por `LCLid`.

Esta función calcula por cliente:

- Consumo total.
- Consumo por periodo del día.
- Consumo en fin de semana y entre semana.
- Promedio y desviación estándar del consumo diario.
- Consumo máximo diario.
- Perfil promedio de 48 medias horas, útil para detectar hora dominante y picos.


In [5]:
def procesar_bloque_hhblock(path, period_cols):
    """Procesa un bloque hhblock y devuelve agregados por LCLid."""
    df = pd.read_csv(path)
    df["day"] = pd.to_datetime(df["day"], errors="coerce")
    hh_cols_local = [col for col in df.columns if col.startswith("hh_")]
    df[hh_cols_local] = df[hh_cols_local].apply(pd.to_numeric, errors="coerce").fillna(0)

    # Consumo diario total por fila LCLid-día
    df["daily_total"] = df[hh_cols_local].sum(axis=1)
    df["is_weekend"] = df["day"].dt.dayofweek >= 5

    # Consumo por periodo en cada fila
    for periodo, cols in period_cols.items():
        df[f"consumo_{periodo}"] = df[cols].sum(axis=1)

    # Agregados de suma por cliente
    agg_sum = df.groupby("LCLid").agg(
        total_consumo=("daily_total", "sum"),
        consumo_madrugada=("consumo_madrugada", "sum"),
        consumo_manana=("consumo_manana", "sum"),
        consumo_tarde=("consumo_tarde", "sum"),
        consumo_noche=("consumo_noche", "sum"),
        dias_observados=("day", "nunique"),
    )

    # Fin de semana / entre semana
    weekend_sum = df.pivot_table(
        index="LCLid",
        columns="is_weekend",
        values="daily_total",
        aggfunc="sum",
        fill_value=0,
    )
    weekend_sum = weekend_sum.rename(columns={False: "consumo_entre_semana", True: "consumo_fin_semana"})
    for col in ["consumo_entre_semana", "consumo_fin_semana"]:
        if col not in weekend_sum.columns:
            weekend_sum[col] = 0
    weekend_sum = weekend_sum[["consumo_entre_semana", "consumo_fin_semana"]]

    # Estadísticos diarios por cliente dentro del bloque
    daily_stats = df.groupby("LCLid")["daily_total"].agg(
        suma_diaria="sum",
        suma_cuadrados_diaria=lambda x: np.square(x).sum(),
        max_diario="max",
        num_dias="count",
    )

    # Perfil intradía: suma por cada media hora para luego consolidar entre bloques
    hh_sum = df.groupby("LCLid")[hh_cols_local].sum()
    hh_sum.columns = [f"sum_{col}" for col in hh_sum.columns]

    salida = agg_sum.join(weekend_sum, how="left").join(daily_stats, how="left").join(hh_sum, how="left")
    salida = salida.fillna(0)
    return salida


# 6. Procesamiento de todos los bloques

Se procesan todos los archivos `hhblock`. Si estás probando la libreta por primera vez, puedes activar `MAX_BLOCKS` con un número pequeño. Para el resultado final, déjalo en `None`.


In [6]:
MAX_BLOCKS = None  # Ejemplo para prueba rápida: 3. Para corrida final: None.
files_to_process = hhblock_files if MAX_BLOCKS is None else hhblock_files[:MAX_BLOCKS]

agregados = []
for i, path in enumerate(files_to_process, start=1):
    print(f"Procesando {i}/{len(files_to_process)}: {path.name}")
    agregados.append(procesar_bloque_hhblock(path, period_cols))

# Consolidación final por cliente
raw_features = pd.concat(agregados).groupby(level=0).sum()
raw_features.index.name = "LCLid"

print("Shape raw_features:", raw_features.shape)
display(raw_features.head())


Procesando 1/112: block_0.csv
Procesando 2/112: block_1.csv
Procesando 3/112: block_10.csv
Procesando 4/112: block_100.csv
Procesando 5/112: block_101.csv
Procesando 6/112: block_102.csv
Procesando 7/112: block_103.csv
Procesando 8/112: block_104.csv
Procesando 9/112: block_105.csv
Procesando 10/112: block_106.csv
Procesando 11/112: block_107.csv
Procesando 12/112: block_108.csv
Procesando 13/112: block_109.csv
Procesando 14/112: block_11.csv
Procesando 15/112: block_110.csv
Procesando 16/112: block_111.csv
Procesando 17/112: block_12.csv
Procesando 18/112: block_13.csv
Procesando 19/112: block_14.csv
Procesando 20/112: block_15.csv
Procesando 21/112: block_16.csv
Procesando 22/112: block_17.csv
Procesando 23/112: block_18.csv
Procesando 24/112: block_19.csv
Procesando 25/112: block_2.csv
Procesando 26/112: block_20.csv
Procesando 27/112: block_21.csv
Procesando 28/112: block_22.csv
Procesando 29/112: block_23.csv
Procesando 30/112: block_24.csv
Procesando 31/112: block_25.csv
Procesan

,total_consumo,consumo_madrugada,consumo_manana,consumo_tarde,consumo_noche,dias_observados,consumo_entre_semana,consumo_fin_semana,suma_diaria,suma_cuadrados_diaria,max_diario,num_dias,sum_hh_0,sum_hh_1,sum_hh_2,sum_hh_3,sum_hh_4,sum_hh_5,sum_hh_6,sum_hh_7,sum_hh_8,sum_hh_9,sum_hh_10,sum_hh_11,sum_hh_12,sum_hh_13,sum_hh_14,sum_hh_15,sum_hh_16,sum_hh_17,sum_hh_18,sum_hh_19,sum_hh_20,sum_hh_21,sum_hh_22,sum_hh_23,sum_hh_24,sum_hh_25,sum_hh_26,sum_hh_27,sum_hh_28,sum_hh_29,sum_hh_30,sum_hh_31,sum_hh_32,sum_hh_33,sum_hh_34,sum_hh_35,sum_hh_36,sum_hh_37,sum_hh_38,sum_hh_39,sum_hh_40,sum_hh_41,sum_hh_42,sum_hh_43,sum_hh_44,sum_hh_45,sum_hh_46,sum_hh_47
LCLid,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
MAC000002,"6,023.6220","1,026.8190","1,193.6640","1,423.4500","2,379.6890",498,"4,172.6620","1,850.9600","6,023.6220","82,683.5796",39.2840,498,146.7090,135.0340,126.0400,106.6700,92.6980,73.5830,64.9140,58.4020,56.9550,55.5690,54.8660,55.3790,54.3320,54.7240,55.6590,62.4900,67.9020,92.5500,109.5760,131.7490,138.5860,142.6200,141.0530,142.4230,140.2400,143.2590,134.2560,130.0960,116.3550,109.9400,101.7750,97.9920,99.3760,104.1200,114.3810,131.6600,149.3560,171.9680,257.7040,220.2780,306.0740,207.5830,186.6210,189.2910,185.3490,181.2720,168.7300,155.4630
MAC000003,"13,998.9640","8,217.5840","2,774.9800","1,507.7780","1,498.6220",735,"10,056.2910","3,942.6730","13,998.9640","364,558.0670",48.2690,735,731.8360,"1,306.9360","1,136.2510",692.5150,608.3870,588.8410,518.5870,603.8210,524.0880,505.2720,528.8060,472.2440,502.4060,531.4040,319.7190,154.6740,165.2720,158.7090,153.5830,142.5140,142.2240,147.7660,164.3050,192.4040,191.1410,181.3890,154.7120,128.0030,108.2520,94.7160,86.1610,87.7990,97.3850,110.9740,125.1810,142.0650,150.5800,158.2840,151.5800,145.1440,150.0240,130.7450,119.6030,111.5960,106.4460,104.3520,89.5670,80.7010
MAC000004,"1,115.7380",269.4850,288.0290,279.4840,278.7400,657,792.3940,323.3440,"1,115.7380","2,011.3300",7.3540,657,21.8100,20.4450,22.5270,24.9930,23.2590,22.0120,20.7580,20.9000,27.4210,21.0400,25.9550,18.3650,20.2090,26.0310,22.8390,27.6600,21.8330,25.5570,23.0750,28.7800,23.2250,22.3000,25.1140,21.4060,24.9000,26.4070,22.9900,20.3470,22.5330,22.8150,25.1590,23.7450,21.0850,21.4130,22.5450,25.5450,23.7830,21.6720,21.7280,22.2670,24.0470,26.3100,24.7490,23.9340,20.7590,22.5890,25.1300,21.7720
MAC000005,"2,890.1850",266.5370,637.4440,885.6620,"1,100.5420",631,"2,059.7830",830.4020,"2,890.1850","14,567.9631",16.7080,631,28.2670,22.7660,21.6740,21.3880,21.1530,21.2120,21.1130,20.9980,20.7710,21.1940,21.8600,24.1410,26.7470,35.0830,61.8440,62.9510,75.8700,79.1490,56.0880,42.8220,48.3480,43.4900,49.9430,55.1090,60.6680,63.3080,70.7450,65.1790,55.2980,52.8010,52.2830,55.2180,61.9910,78.5170,119.1360,150.5180,168.8340,123.1810,86.5840,79.6410,78.3630,83.2720,102.5470,106.8290,93.5370,80.0560,57.7350,39.9630
MAC000006,"2,156.8400",279.8740,681.9520,515.0390,679.9750,755,"1,565.9910",590.8490,"2,156.8400","6,738.5841",6.6360,755,26.8770,24.0840,22.6560,22.9930,22.1770,22.1510,22.2480,22.2280,21.6180,23.4410,23.9000,25.5010,31.2730,33.8300,41.9960,59.1790,76.2730,83.1780,75.4800,77.9900,53.4150,55.0680,48.6820,45.5880,45.7630,44.4700,40.3850,38.5830,38.5120,39.8200,38.8700,41.6550,39.2890,44.1430,47.1400,56.4090,56.3600,70.4350,67.1840,68.8220,67.2400,66.9490,65.4260,56.7210,50.2230,44.1800,36.3700,30.0650


# 7. Construcción del dataset de comportamiento

Aquí se transforman los agregados en variables comparables entre hogares. La mayoría son proporciones, por lo que el modelo captura comportamiento y no solamente magnitud.


In [7]:
features = pd.DataFrame(index=raw_features.index)

eps = 1e-9
total = raw_features["total_consumo"].replace(0, np.nan)

# Distribución horaria
for periodo in ["madrugada", "manana", "tarde", "noche"]:
    features[f"pct_{periodo}"] = raw_features[f"consumo_{periodo}"] / total

# Fin de semana
features["pct_fin_semana"] = raw_features["consumo_fin_semana"] / total
features["pct_entre_semana"] = raw_features["consumo_entre_semana"] / total
features["ratio_fin_semana_vs_semana"] = raw_features["consumo_fin_semana"] / (raw_features["consumo_entre_semana"] + eps)

# Variabilidad diaria global por cliente usando suma y suma de cuadrados
n = raw_features["num_dias"].replace(0, np.nan)
mean_daily = raw_features["suma_diaria"] / n
var_daily = (raw_features["suma_cuadrados_diaria"] / n) - np.square(mean_daily)
std_daily = np.sqrt(np.maximum(var_daily, 0))
features["cv_diario"] = std_daily / (mean_daily + eps)
features["max_diario_vs_promedio"] = raw_features["max_diario"] / (mean_daily + eps)

# Perfil intradía promedio por media hora
sum_hh_cols = [col for col in raw_features.columns if col.startswith("sum_hh_")]
hh_profile = raw_features[sum_hh_cols].copy()
hh_profile.columns = [col.replace("sum_", "") for col in hh_profile.columns]
hh_profile = hh_profile.div(raw_features["dias_observados"].replace(0, np.nan), axis=0)

features["ratio_pico_intradia"] = hh_profile.max(axis=1) / (hh_profile.mean(axis=1) + eps)
features["hh_pico_promedio"] = hh_profile.idxmax(axis=1).str.replace("hh_", "", regex=False).astype(int)
features["hora_pico_promedio"] = features["hh_pico_promedio"] // 2
features["dias_observados"] = raw_features["dias_observados"]
features["consumo_total_referencia"] = raw_features["total_consumo"]

features = features.replace([np.inf, -np.inf], np.nan).fillna(0)

print("Shape features:", features.shape)
display(features.head())
display(features.describe().T)


Shape features: (5560, 14)


,pct_madrugada,pct_manana,pct_tarde,pct_noche,pct_fin_semana,pct_entre_semana,ratio_fin_semana_vs_semana,cv_diario,max_diario_vs_promedio,ratio_pico_intradia,hh_pico_promedio,hora_pico_promedio,dias_observados,consumo_total_referencia
LCLid,,,,,,,,,,,,,,
MAC000002,0.1705,0.1982,0.2363,0.3951,0.3073,0.6927,0.4436,0.3672,3.2478,2.4390,40,20,498,"6,023.6220"
MAC000003,0.5870,0.1982,0.1077,0.1071,0.2816,0.7184,0.3921,0.6060,2.5343,4.4813,1,0,735,"13,998.9640"
MAC000004,0.2415,0.2582,0.2505,0.2498,0.2898,0.7102,0.4081,0.2480,4.3304,1.2381,19,9,657,"1,115.7380"
MAC000005,0.0922,0.2206,0.3064,0.3808,0.2873,0.7127,0.4032,0.3170,3.6478,2.8040,36,18,631,"2,890.1850"
MAC000006,0.1298,0.3162,0.2388,0.3153,0.2739,0.7261,0.3773,0.3060,2.3229,1.8511,17,8,755,"2,156.8400"


,count,mean,std,min,25%,50%,75%,max
pct_madrugada,"5,560.0000",0.1497,0.0696,0.0000,0.1117,0.1403,0.1736,0.8517
pct_manana,"5,560.0000",0.2387,0.0524,0.0000,0.2082,0.2355,0.2643,0.8959
pct_tarde,"5,560.0000",0.2727,0.0528,0.0000,0.2427,0.2704,0.3019,0.5656
pct_noche,"5,560.0000",0.3387,0.0662,0.0000,0.2999,0.3376,0.3777,0.7516
pct_fin_semana,"5,560.0000",0.2954,0.0277,0.0000,0.2827,0.2935,0.3079,0.5417
pct_entre_semana,"5,560.0000",0.7045,0.0290,0.0000,0.6921,0.7065,0.7172,0.9540
ratio_fin_semana_vs_semana,"5,560.0000",0.4214,0.0559,0.0000,0.3942,0.4154,0.4449,1.1818
cv_diario,"5,560.0000",0.3826,0.3851,0.0000,0.2486,0.3263,0.4439,16.9159
max_diario_vs_promedio,"5,560.0000",2.9396,7.7173,0.0000,1.9839,2.4176,3.1064,404.7752
ratio_pico_intradia,"5,560.0000",1.9580,0.7072,0.0000,1.5833,1.8060,2.1297,13.5205


# 8. Variables finales para clustering

Se excluyen variables de referencia como `consumo_total_referencia` y `dias_observados`. También se excluye `pct_entre_semana` porque es complementaria de `pct_fin_semana`, y mantener ambas puede generar redundancia.


In [8]:
variables_cluster_horarios = [
    "pct_madrugada",
    "pct_manana",
    "pct_tarde",
    "pct_noche",
    "cv_diario"
]

variables_cluster_semanal = [
    "cv_diario",
    "pct_fin_semana",
    "ratio_fin_semana_vs_semana",
    "max_diario_vs_promedio"
]

variables_cluster_tipo_consumo = [
    "cv_diario",
    "ratio_pico_intradia",
    "hora_pico_promedio",
    "max_diario_vs_promedio"
]

X_behavior_horario = features[variables_cluster_horarios].copy()
X_behavior_semanal = features[variables_cluster_semanal].copy()
X_behavior_tipo_consumo = features[variables_cluster_tipo_consumo].copy()

scaler = StandardScaler()
X_scaled_horario = scaler.fit_transform(X_behavior_horario)
X_scaled_semanal = scaler.fit_transform(X_behavior_semanal)
X_scaled_tipo_consumo = scaler.fit_transform(X_behavior_tipo_consumo)

print("Variables usadas en clustering por horario:")
print(variables_cluster_horarios)
print("Shape X_scaled_horario:", X_scaled_horario.shape)
print("Variables usadas en clustering por semana:")
print(variables_cluster_semanal)
print("Shape X_scaled_semanal:", X_scaled_semanal.shape)
print("Variables usadas en clustering por tipo de consumo:")
print(variables_cluster_tipo_consumo)
print("Shape X_scaled_tipo_consumo:", X_scaled_tipo_consumo.shape)


Variables usadas en clustering por horario:
['pct_madrugada', 'pct_manana', 'pct_tarde', 'pct_noche', 'cv_diario']
Shape X_scaled_horario: (5560, 5)
Variables usadas en clustering por semana:
['cv_diario', 'pct_fin_semana', 'ratio_fin_semana_vs_semana', 'max_diario_vs_promedio']
Shape X_scaled_semanal: (5560, 4)
Variables usadas en clustering por tipo de consumo:
['cv_diario', 'ratio_pico_intradia', 'hora_pico_promedio', 'max_diario_vs_promedio']
Shape X_scaled_tipo_consumo: (5560, 4)


# 9. Evaluación de K-Means

Se evalúan varios valores de `K`. La decisión final no debe depender únicamente de las métricas, sino también de la interpretación de los centroides y del valor de negocio de cada segmento.


In [9]:
def evaluar_kmeans(X_scaled, Tag):
    resultados = []
    modelos = {}

    for k in range(2, 9):
        kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20, max_iter=300)
        labels = kmeans.fit_predict(X_scaled)
        modelos[k] = kmeans
        distribucion = pd.Series(labels).value_counts(normalize=True) * 100

        resultados.append({
            "K": k,
            "Inertia": kmeans.inertia_,
            "Silhouette": silhouette_score(X_scaled, labels),
            "Davies_Bouldin": davies_bouldin_score(X_scaled, labels),
            "Calinski_Harabasz": calinski_harabasz_score(X_scaled, labels),
            "Cluster_mayor_%": distribucion.max(),
            "Cluster_menor_%": distribucion.min()
        })

    print(f"Resultados KMeans para {Tag}:")
    resultados_kmeans = pd.DataFrame(resultados)
    display(resultados_kmeans)
    resultados_kmeans.to_csv(TABLES_DIR / f"evaluacion_kmeans_comportamiento_{Tag}.csv", index=False)
    return resultados_kmeans, modelos

resultados_kmeans_por_tag = {}
modelos_kmeans_por_tag = {}

resultados_kmeans_por_tag["horarios"], modelos_kmeans_por_tag["horarios"] = evaluar_kmeans(X_scaled_horario, "horarios")
resultados_kmeans_por_tag["semanal"], modelos_kmeans_por_tag["semanal"] = evaluar_kmeans(X_scaled_semanal, "semanal")
resultados_kmeans_por_tag["tipo_consumo"], modelos_kmeans_por_tag["tipo_consumo"] = evaluar_kmeans(X_scaled_tipo_consumo, "tipo_consumo")


Resultados KMeans para horarios:


,K,Inertia,Silhouette,Davies_Bouldin,Calinski_Harabasz,Cluster_mayor_%,Cluster_menor_%
0,2,"22,642.4303",0.2279,1.5642,"1,266.0220",52.1583,47.8417
1,3,"18,731.9152",0.2439,1.2321,"1,345.0666",51.5108,1.3489
2,4,"15,058.0677",0.2461,0.8850,"1,567.1393",51.5827,0.0360
3,5,"12,694.4598",0.2327,0.9626,"1,652.5314",39.5683,0.0360
4,6,"10,944.5729",0.2325,0.9398,"1,710.7128",29.1727,0.0360
5,7,"9,945.2896",0.2055,0.9991,"1,661.5507",33.4353,0.0360
6,8,"9,203.2551",0.1959,1.0439,"1,602.6858",24.2086,0.0360


Resultados KMeans para semanal:


,K,Inertia,Silhouette,Davies_Bouldin,Calinski_Harabasz,Cluster_mayor_%,Cluster_menor_%
0,2,"13,058.4936",0.9758,0.0393,"3,907.8636",99.9640,0.0360
1,3,"7,858.2804",0.4453,0.5652,"5,085.0919",69.3705,0.0360
2,4,"5,721.5856",0.4427,0.6506,"5,346.8462",67.8417,0.0360
3,5,"4,447.5326",0.3923,0.6617,"5,555.7304",51.3129,0.0360
4,6,"3,661.5382",0.3821,0.7214,"5,636.1978",49.8022,0.0360
5,7,"3,150.8493",0.4010,0.6980,"5,607.1481",45.9532,0.0360
6,8,"2,690.9500",0.3794,0.6961,"5,762.0147",35.9712,0.0360


Resultados KMeans para tipo_consumo:


,K,Inertia,Silhouette,Davies_Bouldin,Calinski_Harabasz,Cluster_mayor_%,Cluster_menor_%
0,2,"13,233.0978",0.9750,0.0395,"3,782.9663",99.9640,0.0360
1,3,"8,566.8508",0.5683,0.6589,"4,434.6356",82.8777,0.0360
2,4,"6,209.4622",0.5508,0.5981,"4,781.1800",81.9604,0.0360
3,5,"5,085.1969",0.3646,0.7715,"4,684.9261",60.4856,0.0360
4,6,"4,429.6846",0.3726,0.7561,"4,466.1787",55.8273,0.0360
5,7,"3,924.0909",0.3726,0.7903,"4,319.8548",54.9101,0.0360
6,8,"3,528.1383",0.3365,0.8357,"4,206.5388",48.9209,0.0360


# 10. Gráficas de evaluación de K-Means

Esta sección visualiza los resultados de K-Means para los tres enfoques de variables: horario, semanal y tipo de consumo. Las gráficas permiten comparar separación, compacidad y balance de clusters antes de seleccionar un valor de `K`.


In [10]:
def cargar_resultados_kmeans_para_graficas():
    if "resultados_kmeans_por_tag" in globals() and resultados_kmeans_por_tag:
        return resultados_kmeans_por_tag

    resultados = {}
    for tag in ["horarios", "semanal", "tipo_consumo"]:
        path = TABLES_DIR / f"evaluacion_kmeans_comportamiento_{tag}.csv"
        if path.exists():
            resultados[tag] = pd.read_csv(path)
        else:
            raise FileNotFoundError(f"No se encontró {path}. Ejecuta primero la sección 9.")
    return resultados

resultados_kmeans_graficas = cargar_resultados_kmeans_para_graficas()
metricas_principales = ["Inertia", "Silhouette", "Davies_Bouldin", "Calinski_Harabasz"]

for tag, df_resultados in resultados_kmeans_graficas.items():
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.flatten()

    for ax, metrica in zip(axes, metricas_principales):
        ax.plot(df_resultados["K"], df_resultados[metrica], marker="o", color="#2F6F9F")
        ax.set_title(f"{tag}: {metrica}")
        ax.set_xlabel("K")
        ax.set_ylabel(metrica)
        ax.grid(alpha=0.25)

    axes[4].plot(df_resultados["K"], df_resultados["Cluster_mayor_%"], marker="o", label="Cluster mayor", color="#C75D2C")
    axes[4].plot(df_resultados["K"], df_resultados["Cluster_menor_%"], marker="o", label="Cluster menor", color="#5B8C5A")
    axes[4].set_title(f"{tag}: balance de clusters")
    axes[4].set_xlabel("K")
    axes[4].set_ylabel("Porcentaje de hogares")
    axes[4].grid(alpha=0.25)
    axes[4].legend()

    axes[5].axis("off")
    fig.suptitle(f"Evaluación K-Means - enfoque {tag}", fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(IMAGES_DIR / f"kmeans_resultados_{tag}.png", dpi=160, bbox_inches="tight")
    plt.show()


# 11. Comparación gráfica entre enfoques de K-Means

Se comparan los tres enfoques de variables para identificar si alguno ofrece mejor separación o mejor balance de clusters. Esta comparación no selecciona automáticamente el modelo final; solo resume evidencia visual.


In [11]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
metricas_comparacion = ["Silhouette", "Davies_Bouldin", "Calinski_Harabasz", "Cluster_mayor_%"]
colores = {"horarios": "#2F6F9F", "semanal": "#C75D2C", "tipo_consumo": "#5B8C5A"}

for ax, metrica in zip(axes.flatten(), metricas_comparacion):
    for tag, df_resultados in resultados_kmeans_graficas.items():
        ax.plot(df_resultados["K"], df_resultados[metrica], marker="o", label=tag, color=colores.get(tag))
    ax.set_title(f"Comparación: {metrica}")
    ax.set_xlabel("K")
    ax.set_ylabel(metrica)
    ax.grid(alpha=0.25)

axes[0, 0].legend()
fig.suptitle("Comparación de K-Means por enfoque de variables", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(IMAGES_DIR / "kmeans_comparacion_enfoques.png", dpi=160, bbox_inches="tight")
plt.show()

registros_mejores = []
for tag, df_resultados in resultados_kmeans_graficas.items():
    idx_silhouette = df_resultados["Silhouette"].idxmax()
    idx_davies = df_resultados["Davies_Bouldin"].idxmin()
    idx_balance = (df_resultados["Cluster_mayor_%"] - df_resultados["Cluster_menor_%"]).idxmin()

    registros_mejores.append({"enfoque": tag, "criterio": "max_silhouette", **df_resultados.loc[idx_silhouette].to_dict()})
    registros_mejores.append({"enfoque": tag, "criterio": "min_davies_bouldin", **df_resultados.loc[idx_davies].to_dict()})
    registros_mejores.append({"enfoque": tag, "criterio": "mejor_balance", **df_resultados.loc[idx_balance].to_dict()})

df_11_resumen_mejores_kmeans = pd.DataFrame(registros_mejores)
df_11_resumen_mejores_kmeans.to_csv(TABLES_DIR / "resumen_mejores_kmeans_por_enfoque.csv", index=False)
display(df_11_resumen_mejores_kmeans)


,enfoque,criterio,K,Inertia,Silhouette,Davies_Bouldin,Calinski_Harabasz,Cluster_mayor_%,Cluster_menor_%
0,horarios,max_silhouette,4.0000,"15,058.0677",0.2461,0.8850,"1,567.1393",51.5827,0.0360
1,horarios,min_davies_bouldin,4.0000,"15,058.0677",0.2461,0.8850,"1,567.1393",51.5827,0.0360
2,horarios,mejor_balance,2.0000,"22,642.4303",0.2279,1.5642,"1,266.0220",52.1583,47.8417
3,semanal,max_silhouette,2.0000,"13,058.4936",0.9758,0.0393,"3,907.8636",99.9640,0.0360
4,semanal,min_davies_bouldin,2.0000,"13,058.4936",0.9758,0.0393,"3,907.8636",99.9640,0.0360
5,semanal,mejor_balance,8.0000,"2,690.9500",0.3794,0.6961,"5,762.0147",35.9712,0.0360
6,tipo_consumo,max_silhouette,2.0000,"13,233.0978",0.9750,0.0395,"3,782.9663",99.9640,0.0360
7,tipo_consumo,min_davies_bouldin,2.0000,"13,233.0978",0.9750,0.0395,"3,782.9663",99.9640,0.0360
8,tipo_consumo,mejor_balance,8.0000,"3,528.1383",0.3365,0.8357,"4,206.5388",48.9209,0.0360


# 12. Decisión del número de clusters

Se selecciona **K = 3** para los tres enfoques de comportamiento, ya que ofrece el mejor equilibrio entre desempeño, interpretabilidad y utilidad de negocio. Aunque algunos valores de K mayores presentan mejoras puntuales en métricas como Davies-Bouldin o Calinski-Harabasz, también generan clusters extremadamente pequeños, lo que puede indicar sobrefragmentación o aislamiento de valores atípicos.

En los enfoques semanal y tipo de consumo, K = 2 muestra métricas internas muy altas; sin embargo, el balance de clusters evidencia que prácticamente todos los hogares se concentran en un solo grupo, por lo que no resulta útil para segmentación. Por esta razón, K = 3 permite construir perfiles más accionables y comparables entre dimensiones: horario, patrón semanal y tipo de consumo.


In [12]:
decision_k3 = """
Decisión del modelo de comportamiento
-------------------------------------
Se selecciona K = 3 para los tres enfoques de comportamiento, ya que ofrece el mejor equilibrio entre desempeño, interpretabilidad y utilidad de negocio. Aunque algunos valores de K mayores presentan mejoras puntuales en métricas como Davies-Bouldin o Calinski-Harabasz, también generan clusters extremadamente pequeños, lo que puede indicar sobrefragmentación o aislamiento de valores atípicos.

En los enfoques semanal y tipo de consumo, K = 2 muestra métricas internas muy altas; sin embargo, el balance de clusters evidencia que prácticamente todos los hogares se concentran en un solo grupo, por lo que no resulta útil para segmentación. Por esta razón, K = 3 permite construir perfiles más accionables y comparables entre dimensiones: horario, patrón semanal y tipo de consumo.
"""
print(decision_k3)

df_12_decision_k3 = pd.DataFrame({
    "dimension": ["horario", "semanal", "tipo_consumo"],
    "modelo_seleccionado": ["KMeans"] * 3,
    "K_seleccionado": [3, 3, 3],
    "criterio": [
        "Interpretabilidad del horario dominante de consumo",
        "Balance entre patrón semanal y estabilidad de consumo",
        "Separación accionable de intensidad y concentración de picos"
    ],
    "justificacion": [
        "K=3 permite distinguir perfiles horarios sin sobrefragmentar hogares.",
        "K=2 concentra demasiados hogares en un solo grupo; K=3 genera segmentos más útiles.",
        "K=3 conserva interpretabilidad y evita clusters demasiado pequeños frente a K mayores."
    ]
})

df_12_decision_k3.to_csv(TABLES_DIR / "df_12_decision_k3.csv", index=False)
display(df_12_decision_k3)



Decisión del modelo de comportamiento
-------------------------------------
Se selecciona K = 3 para los tres enfoques de comportamiento, ya que ofrece el mejor equilibrio entre desempeño, interpretabilidad y utilidad de negocio. Aunque algunos valores de K mayores presentan mejoras puntuales en métricas como Davies-Bouldin o Calinski-Harabasz, también generan clusters extremadamente pequeños, lo que puede indicar sobrefragmentación o aislamiento de valores atípicos.

En los enfoques semanal y tipo de consumo, K = 2 muestra métricas internas muy altas; sin embargo, el balance de clusters evidencia que prácticamente todos los hogares se concentran en un solo grupo, por lo que no resulta útil para segmentación. Por esta razón, K = 3 permite construir perfiles más accionables y comparables entre dimensiones: horario, patrón semanal y tipo de consumo.



,dimension,modelo_seleccionado,K_seleccionado,criterio,justificacion
0,horario,KMeans,3,Interpretabilidad del horario dominante de consumo,K=3 permite distinguir perfiles horarios sin sobrefragmentar hogares.
1,semanal,KMeans,3,Balance entre patrón semanal y estabilidad de consumo,K=2 concentra demasiados hogares en un solo grupo; K=3 genera segmentos más útiles.
2,tipo_consumo,KMeans,3,Separación accionable de intensidad y concentración de picos,K=3 conserva interpretabilidad y evita clusters demasiado pequeños frente a K mayores.


# 13. Entrenamiento final con K = 3

Se entrena K-Means con K = 3 para cada enfoque. Las etiquetas numéricas se transforman en nombres descriptivos para que el resultado final pueda leerse como perfiles de negocio y no sólo como números de cluster.


In [13]:
def entrenar_kmeans_final(X_scaled, X_original, variables, enfoque, k=3):
    modelo = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20, max_iter=300)
    labels = modelo.fit_predict(X_scaled)

    df_labels = pd.DataFrame({
        "LCLid": features.index,
        f"cluster_{enfoque}": labels
    })

    df_perfil = X_original.copy()
    df_perfil[f"cluster_{enfoque}"] = labels
    perfil = df_perfil.groupby(f"cluster_{enfoque}")[variables].mean().reset_index()
    distribucion = df_labels[f"cluster_{enfoque}"].value_counts().sort_index().rename_axis(f"cluster_{enfoque}").reset_index(name="hogares")
    distribucion["pct_hogares"] = distribucion["hogares"] / len(df_labels) * 100
    perfil = perfil.merge(distribucion, on=f"cluster_{enfoque}", how="left")

    metricas = pd.DataFrame([{
        "enfoque": enfoque,
        "modelo": "KMeans",
        "K": k,
        "Inertia": modelo.inertia_,
        "Silhouette": silhouette_score(X_scaled, labels),
        "Davies_Bouldin": davies_bouldin_score(X_scaled, labels),
        "Calinski_Harabasz": calinski_harabasz_score(X_scaled, labels),
        "Cluster_mayor_%": distribucion["pct_hogares"].max(),
        "Cluster_menor_%": distribucion["pct_hogares"].min()
    }])

    return modelo, labels, perfil, distribucion, metricas

modelo_horario_k3, labels_horario_k3, perfil_horario_k3, distribucion_horario_k3, metricas_horario_k3 = entrenar_kmeans_final(
    X_scaled_horario, X_behavior_horario, variables_cluster_horarios, "horario", k=3
)
modelo_semanal_k3, labels_semanal_k3, perfil_semanal_k3, distribucion_semanal_k3, metricas_semanal_k3 = entrenar_kmeans_final(
    X_scaled_semanal, X_behavior_semanal, variables_cluster_semanal, "semanal", k=3
)
modelo_tipo_consumo_k3, labels_tipo_consumo_k3, perfil_tipo_consumo_k3, distribucion_tipo_consumo_k3, metricas_tipo_consumo_k3 = entrenar_kmeans_final(
    X_scaled_tipo_consumo, X_behavior_tipo_consumo, variables_cluster_tipo_consumo, "tipo_consumo", k=3
)

df_13_metricas_finales_k3 = pd.concat([metricas_horario_k3, metricas_semanal_k3, metricas_tipo_consumo_k3], ignore_index=True)
df_13_metricas_finales_k3.to_csv(TABLES_DIR / "df_13_metricas_finales_k3.csv", index=False)

display(df_13_metricas_finales_k3)


,enfoque,modelo,K,Inertia,Silhouette,Davies_Bouldin,Calinski_Harabasz,Cluster_mayor_%,Cluster_menor_%
0,horario,KMeans,3,"18,731.9152",0.2439,1.2321,"1,345.0666",51.5108,1.3489
1,semanal,KMeans,3,"7,858.2804",0.4453,0.5652,"5,085.0919",69.3705,0.0360
2,tipo_consumo,KMeans,3,"8,566.8508",0.5683,0.6589,"4,434.6356",82.8777,0.0360


# 14. Perfil y nombre de los clusters

Para que el resultado sea interpretable, se asigna una etiqueta descriptiva a cada cluster de acuerdo con el comportamiento promedio observado en sus variables principales.


In [17]:
def etiquetar_horario(perfil):
    share_cols = ["pct_madrugada", "pct_manana", "pct_tarde", "pct_noche"]
    nombres = {}
    for _, row in perfil.iterrows():
        cluster = int(row["cluster_horario"])
        periodo = max(share_cols, key=lambda col: row[col]).replace("pct_", "")
        cv = row.get("cv_diario", 0)
        sufijo = "variable" if cv >= perfil["cv_diario"].median() else "estable"
        nombres[cluster] = f"consumo_{periodo}_{sufijo}"
    return nombres


def etiquetar_semanal(perfil):
    nombres = {}
    mediana_fin_semana = perfil["pct_fin_semana"].median()
    mediana_cv = perfil["cv_diario"].median()
    for _, row in perfil.iterrows():
        cluster = int(row["cluster_semanal"])
        if row["pct_fin_semana"] >= mediana_fin_semana:
            base = "consumo_fin_semana"
        else:
            base = "consumo_entre_semana"
        sufijo = "variable" if row["cv_diario"] >= mediana_cv else "estable"
        nombres[cluster] = f"{base}_{sufijo}"
    return nombres


def etiquetar_tipo_consumo(perfil):
    nombres = {}
    picos_rank = perfil["ratio_pico_intradia"].rank(method="first")
    cv_rank = perfil["cv_diario"].rank(method="first")
    for idx, row in perfil.iterrows():
        cluster = int(row["cluster_tipo_consumo"])
        intensidad = "picos_altos" if picos_rank.loc[idx] == picos_rank.max() else ("picos_bajos" if picos_rank.loc[idx] == picos_rank.min() else "picos_moderados")
        variacion = "variabilidad_alta" if cv_rank.loc[idx] == cv_rank.max() else ("variabilidad_baja" if cv_rank.loc[idx] == cv_rank.min() else "variabilidad_media")
        nombres[cluster] = f"{intensidad}_{variacion}"
    return nombres

map_horario = etiquetar_horario(perfil_horario_k3)
map_semanal = etiquetar_semanal(perfil_semanal_k3)
map_tipo_consumo = etiquetar_tipo_consumo(perfil_tipo_consumo_k3)

perfil_horario_k3["horario_consumo"] = perfil_horario_k3["cluster_horario"].map(map_horario)
perfil_semanal_k3["semanal"] = perfil_semanal_k3["cluster_semanal"].map(map_semanal)
perfil_tipo_consumo_k3["tipo_consumo"] = perfil_tipo_consumo_k3["cluster_tipo_consumo"].map(map_tipo_consumo)

perfil_horario_k3.to_csv(TABLES_DIR / "df_14_perfil_horario_k3.csv", index=False)
perfil_semanal_k3.to_csv(TABLES_DIR / "df_14_perfil_semanal_k3.csv", index=False)
perfil_tipo_consumo_k3.to_csv(TABLES_DIR / "df_14_perfil_tipo_consumo_k3.csv", index=False)

print("Etiquetas horario:", map_horario)
print("Etiquetas semanal:", map_semanal)
print("Etiquetas tipo de consumo:", map_tipo_consumo)

display(perfil_horario_k3)
display(perfil_semanal_k3)
display(perfil_tipo_consumo_k3)


Etiquetas horario: {0: 'consumo_noche_variable', 1: 'consumo_tarde_estable', 2: 'consumo_madrugada_variable'}
Etiquetas semanal: {0: 'consumo_fin_semana_estable', 1: 'consumo_fin_semana_variable', 2: 'consumo_entre_semana_variable'}
Etiquetas tipo de consumo: {0: 'picos_bajos_variabilidad_baja', 1: 'picos_moderados_variabilidad_media', 2: 'picos_altos_variabilidad_alta'}


,cluster_horario,pct_madrugada,pct_manana,pct_tarde,pct_noche,cv_diario,hogares,pct_hogares,horario_consumo
0,0,0.1522,0.2131,0.2526,0.3817,0.3809,2864,51.5108,consumo_noche_variable
1,1,0.1354,0.2692,0.2987,0.2966,0.3647,2621,47.1403,consumo_tarde_estable
2,2,0.5545,0.1513,0.1298,0.1644,1.0732,75,1.3489,consumo_madrugada_variable


,cluster_semanal,cv_diario,pct_fin_semana,ratio_fin_semana_vs_semana,max_diario_vs_promedio,hogares,pct_hogares,semanal
0,0,0.3748,0.2830,0.3957,2.7680,3857,69.3705,consumo_fin_semana_estable
1,1,0.3808,0.3236,0.4799,2.8609,1701,30.5935,consumo_fin_semana_variable
2,2,16.8876,0.0821,0.0910,400.7445,2,0.0360,consumo_entre_semana_variable


,cluster_tipo_consumo,cv_diario,ratio_pico_intradia,hora_pico_promedio,max_diario_vs_promedio,hogares,pct_hogares,tipo_consumo
0,0,0.3670,1.8853,19.0501,2.7572,4608,82.8777,picos_bajos_variabilidad_baja
1,1,0.4236,2.3065,7.7305,2.9867,950,17.0863,picos_moderados_variabilidad_media
2,2,16.8876,3.7439,14.0000,400.7445,2,0.0360,picos_altos_variabilidad_alta


## Análisis de perfiles de consumo por horario


A partir de la segmentación por comportamiento horario, se identificaron tres clusters con patrones diferenciados en la distribución del consumo a lo largo del día.

### Cluster 0: Consumo nocturno residencial

Este grupo presenta la mayor concentración de consumo durante la noche (38.17%), seguido de la tarde (25.26%) y la mañana (21.31%). El consumo en madrugada es relativamente bajo (15.22%).

Este patrón sugiere hogares donde la actividad eléctrica se intensifica al final del día, posiblemente asociado a rutinas típicas residenciales como uso de electrodomésticos, iluminación y entretenimiento en horario nocturno.

---

### Cluster 1: Consumo diurno / equilibrado

El consumo en este cluster se distribuye de manera más uniforme entre la mañana (26.92%), la tarde (29.87%) y la noche (29.66%), con una menor participación en madrugada (13.54%).

Este perfil representa consumidores con hábitos más estables y regulares a lo largo del día, sin una concentración marcada en un horario específico. Puede asociarse a hogares con presencia constante o actividades distribuidas durante el día.

---

### Cluster 2: Consumo atípico en madrugada

Este cluster presenta un comportamiento claramente diferenciado, con un 55.45% del consumo concentrado en la madrugada, mientras que el resto del día tiene participaciones considerablemente menores.

Además, este grupo muestra la mayor variabilidad diaria (`cv_diario = 1.0732`), lo que indica un patrón irregular y poco consistente.

Este tipo de comportamiento puede estar asociado a:
- Consumo automatizado
- Equipos funcionando en horarios nocturnos
- Actividades no convencionales
- Posibles casos atípicos o anomalías

---

### Conclusión

La segmentación por horario permite identificar tres perfiles principales de consumo:

- **Cluster 0:** Consumidores nocturnos residenciales  
- **Cluster 1:** Consumidores diurnos o equilibrados  
- **Cluster 2:** Consumidores atípicos con alta actividad en madrugada  

Estos perfiles no solo reflejan cuándo consumen los usuarios, sino que también aportan una base sólida para generar estrategias de negocio, detección de patrones anómalos y recomendaciones personalizadas.

## Análisis de perfiles de consumo semanal


A partir de la segmentación semanal, se identificaron tres clusters con diferencias en la proporción de consumo durante fines de semana, la variabilidad diaria y la presencia de días con consumo máximo atípico.

### Cluster 0: Consumo principalmente entre semana

Este grupo presenta un consumo en fin de semana de 28.30%, con un `ratio_fin_semana_vs_semana` de 0.3957. Esto indica que el consumo se concentra principalmente entre semana.

Su variabilidad diaria es moderada (`cv_diario = 0.3748`) y el valor de `max_diario_vs_promedio = 2.7680` sugiere que puede tener algunos días de mayor consumo, pero sin llegar a niveles extremos.

---

### Cluster 1: Consumo con mayor actividad en fin de semana

Este cluster muestra el mayor porcentaje de consumo en fin de semana, con 32.36%, y el mayor `ratio_fin_semana_vs_semana` entre los grupos principales, con 0.4799.

Esto sugiere hogares donde el consumo aumenta relativamente durante sábado y domingo. Puede asociarse con mayor presencia en casa, actividades domésticas, entretenimiento o uso intensivo de aparatos durante el fin de semana.

---

### Cluster 2: Consumo atípico con baja actividad en fin de semana

Este grupo tiene un comportamiento claramente atípico. Presenta solo 8.21% de consumo en fin de semana y un `ratio_fin_semana_vs_semana` de 0.0910, lo que indica una fuerte concentración del consumo entre semana.

Además, muestra valores extremadamente altos de variabilidad diaria (`cv_diario = 16.8876`) y de `max_diario_vs_promedio = 400.7445`. Esto sugiere la presencia de consumos muy irregulares, posibles registros anómalos o días con picos extraordinarios.

---

### Conclusión

La segmentación semanal permite identificar tres perfiles principales:

- **Cluster 0:** Consumidores principalmente entre semana  
- **Cluster 1:** Consumidores con mayor actividad en fin de semana  
- **Cluster 2:** Consumidores atípicos con picos extremos y baja actividad en fin de semana  

Este enfoque complementa la segmentación horaria, ya que permite distinguir no solo en qué momento del día consumen los usuarios, sino también si su consumo se concentra en días laborales, fines de semana o en patrones irregulares.

## Análisis de perfiles por tipo de consumo

A partir de la segmentación por tipo de consumo, se identificaron tres clusters diferenciados según regularidad diaria, presencia de picos intradía, hora promedio de mayor consumo y máximos diarios frente al promedio.

### Cluster 0: Consumo estable con pico nocturno

Este grupo presenta la menor variabilidad diaria (`cv_diario = 0.3670`) y el menor `ratio_pico_intradia = 1.8853`, lo que indica un consumo relativamente estable.

La hora promedio de mayor consumo es cercana a las 19:00 horas, por lo que puede interpretarse como un perfil residencial con mayor actividad durante la tarde-noche.

---

### Cluster 1: Consumo con picos matutinos moderados

Este cluster presenta una variabilidad ligeramente mayor (`cv_diario = 0.4236`) y un `ratio_pico_intradia = 2.3065`, lo que indica mayor concentración del consumo en ciertos momentos del día.

La hora promedio de mayor consumo es cercana a las 08:00 horas, por lo que puede asociarse con hogares con mayor actividad en la mañana.

---

### Cluster 2: Consumo altamente atípico

Este grupo presenta valores extremadamente altos de variabilidad diaria (`cv_diario = 16.8876`) y de `max_diario_vs_promedio = 400.7445`.

Además, tiene el mayor `ratio_pico_intradia = 3.7439`, lo que indica consumos muy concentrados en momentos específicos. Este comportamiento puede estar relacionado con registros anómalos, consumos extraordinarios o usuarios con patrones altamente irregulares.

---

### Conclusión

La segmentación por tipo de consumo permite identificar tres perfiles principales:

- **Cluster 0:** Consumo estable con pico nocturno  
- **Cluster 1:** Consumo con picos matutinos moderados  
- **Cluster 2:** Consumo altamente atípico  

Este enfoque complementa los análisis horario y semanal, ya que permite distinguir la forma del consumo: usuarios estables, usuarios con picos moderados y usuarios con patrones extremos o irregulares.

# 15. Graficación del entrenamiento final

Se visualizan los clusters finales en dos dimensiones mediante PCA para cada enfoque. Estas gráficas permiten revisar la separación relativa de los segmentos y complementar la lectura de los perfiles promedio.


In [18]:
def graficar_clusters_pca(X_scaled, labels, enfoque, archivo):
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    coords = pca.fit_transform(X_scaled)
    df_plot = pd.DataFrame({
        "PC1": coords[:, 0],
        "PC2": coords[:, 1],
        "cluster": labels
    })

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(df_plot["PC1"], df_plot["PC2"], c=df_plot["cluster"], cmap="tab10", s=18, alpha=0.65)
    plt.title(f"Clusters K-Means K=3 - enfoque {enfoque}")
    plt.xlabel("Componente principal 1")
    plt.ylabel("Componente principal 2")
    plt.colorbar(scatter, label="Cluster")
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / archivo, dpi=160, bbox_inches="tight")
    plt.show()


def graficar_perfil(perfil, cluster_col, label_col, variables, enfoque, archivo):
    df_long = perfil[[cluster_col, label_col] + variables].melt(
        id_vars=[cluster_col, label_col],
        var_name="variable",
        value_name="valor_promedio"
    )

    plt.figure(figsize=(12, 5))
    for etiqueta, datos in df_long.groupby(label_col):
        plt.plot(datos["variable"], datos["valor_promedio"], marker="o", label=etiqueta)
    plt.title(f"Perfil promedio por cluster - {enfoque}")
    plt.xlabel("Variable")
    plt.ylabel("Valor promedio")
    plt.xticks(rotation=35, ha="right")
    plt.legend(title="Perfil", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / archivo, dpi=160, bbox_inches="tight")
    plt.show()

graficar_clusters_pca(X_scaled_horario, labels_horario_k3, "horario", "clusters_kmeans_k3_horario_pca.png")
graficar_clusters_pca(X_scaled_semanal, labels_semanal_k3, "semanal", "clusters_kmeans_k3_semanal_pca.png")
graficar_clusters_pca(X_scaled_tipo_consumo, labels_tipo_consumo_k3, "tipo_consumo", "clusters_kmeans_k3_tipo_consumo_pca.png")

graficar_perfil(perfil_horario_k3, "cluster_horario", "horario_consumo", variables_cluster_horarios, "horario", "perfil_kmeans_k3_horario.png")
graficar_perfil(perfil_semanal_k3, "cluster_semanal", "semanal", variables_cluster_semanal, "semanal", "perfil_kmeans_k3_semanal.png")
graficar_perfil(perfil_tipo_consumo_k3, "cluster_tipo_consumo", "tipo_consumo", variables_cluster_tipo_consumo, "tipo_consumo", "perfil_kmeans_k3_tipo_consumo.png")


# 16. DataFrame final de comportamiento por casa

El entregable final resume, para cada hogar, el perfil asignado en las tres dimensiones de comportamiento: horario de consumo, patrón semanal y tipo de consumo. Este archivo queda disponible para análisis de negocio y posibles campañas o promociones.


In [19]:
df_16_comportamiento_por_casa = pd.DataFrame({
    "casa": features.index,
    "horario_consumo": pd.Series(labels_horario_k3).map(map_horario).values,
    "semanal": pd.Series(labels_semanal_k3).map(map_semanal).values,
    "tipo_consumo": pd.Series(labels_tipo_consumo_k3).map(map_tipo_consumo).values,
})

# Se conserva una versión extendida con etiquetas numéricas para trazabilidad técnica.
df_16_comportamiento_por_casa_detalle = df_16_comportamiento_por_casa.copy()
df_16_comportamiento_por_casa_detalle["cluster_horario"] = labels_horario_k3
df_16_comportamiento_por_casa_detalle["cluster_semanal"] = labels_semanal_k3
df_16_comportamiento_por_casa_detalle["cluster_tipo_consumo"] = labels_tipo_consumo_k3

if HOUSEHOLDS_PATH.exists():
    households = pd.read_csv(HOUSEHOLDS_PATH)
    columnas_hogares = [col for col in ["LCLid", "stdorToU", "Acorn_grouped"] if col in households.columns]
    if "LCLid" in columnas_hogares:
        df_16_comportamiento_por_casa_detalle = df_16_comportamiento_por_casa_detalle.merge(
            households[columnas_hogares].rename(columns={"LCLid": "casa"}),
            on="casa",
            how="left"
        )

df_16_comportamiento_por_casa.to_csv(TABLES_DIR / "df_16_comportamiento_por_casa.csv", index=False)

DF_Comportamiento_cluster = df_16_comportamiento_por_casa.copy()
DF_Comportamiento_cluster.to_csv(TABLES_DIR / "DF_Comportamiento_cluster.csv", index=False)
df_16_comportamiento_por_casa_detalle.to_csv(TABLES_DIR / "df_16_comportamiento_por_casa_detalle.csv", index=False)

print("Shape DF_Comportamiento_cluster:", DF_Comportamiento_cluster.shape)
display(DF_Comportamiento_cluster.head())
display(df_16_comportamiento_por_casa_detalle.head())


Shape DF_Comportamiento_cluster: (5560, 4)


,casa,horario_consumo,semanal,tipo_consumo
0,MAC000002,consumo_noche_variable,consumo_fin_semana_variable,picos_bajos_variabilidad_baja
1,MAC000003,consumo_madrugada_variable,consumo_fin_semana_estable,picos_moderados_variabilidad_media
2,MAC000004,consumo_tarde_estable,consumo_fin_semana_estable,picos_moderados_variabilidad_media
3,MAC000005,consumo_noche_variable,consumo_fin_semana_estable,picos_bajos_variabilidad_baja
4,MAC000006,consumo_tarde_estable,consumo_fin_semana_estable,picos_moderados_variabilidad_media


,casa,horario_consumo,semanal,tipo_consumo,cluster_horario,cluster_semanal,cluster_tipo_consumo,stdorToU,Acorn_grouped
0,MAC000002,consumo_noche_variable,consumo_fin_semana_variable,picos_bajos_variabilidad_baja,0,1,0,Std,Affluent
1,MAC000003,consumo_madrugada_variable,consumo_fin_semana_estable,picos_moderados_variabilidad_media,2,0,1,Std,Adversity
2,MAC000004,consumo_tarde_estable,consumo_fin_semana_estable,picos_moderados_variabilidad_media,1,0,1,Std,Affluent
3,MAC000005,consumo_noche_variable,consumo_fin_semana_estable,picos_bajos_variabilidad_baja,0,0,0,ToU,Affluent
4,MAC000006,consumo_tarde_estable,consumo_fin_semana_estable,picos_moderados_variabilidad_media,1,0,1,Std,Adversity
